# "THE PRICE IS RIGHT" — Week 8, Day 3

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
import logging
import requests
from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("Could not find the repository .env file")
load_dotenv(dotenv_path, override=True)

from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required in the repository .env file")
openai = OpenAI()
MODEL = os.getenv("SCANNER_MODEL", "gpt-5-nano")
print(f"Scanner model: {MODEL}")

/Users/marcolerma/GitHub/applied-llm-engineering/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Scanner model: gpt-5-nano


In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

  0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: 12" 256GB Android 16 Tablet with Keyboard & Stylus for $114 + free shipping\nDetails: A 12" screen with a keyboard and stylus included makes this a reasonable pick for anyone needing a budget laptop replacement for note-taking or browsing. Apply coupon code "XF4CR8P3" for a savings of $126.\xa0 Buy Now at Amazon\nFeatures: 12" 2K 120Hz IPS display Integrated on-device Gemini AI 256GB storage 10,000mAh battery for all-day use Dual-band 5G WiFi and Bluetooth 5.4\nURL: https://www.dealnews.com/12-256-GB-Android-16-Tablet-with-Keyboard-Stylus-for-114-free-shipping/22058390.html?iref=rss-c39'

### Ask GPT-5 nano to summarize deals and identify their prices

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Wegear 70W GaN 3-Port USB C Fast Wall Charger for $25 w/ Prime + free shipping
Details: Compact enough to replace a bulky laptop brick while still pushing 70W across three ports, which suits anyone juggling a phone, tablet, and laptop off one outlet. Apply coupon code "NIFH6GO2" for a total savings of $30. This deal is for Prime members only. Buy Now at Amazon
Features: 70W PD3

In [8]:
response = openai.chat.completions.parse(
    model=MODEL,
    messages=messages,
    response_format=DealSelection,
)
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='The SmallRig 68.9" Carbon Fiber Video Monopod is a rugged, lightweight support designed for stable handheld or mounted shots. It features a collapsible four-section design for compact transport, and dual 1/4-inch-20 and 3/8-inch-16 mounting compatibility to fit a wide range of cameras and accessories. It offers a one-button lift and lock height adjustment plus quick-release feet for rapid setup, enabling quick, confident adjustments on location. Built from carbon fiber, this monopod combines strength with lightness for demanding video work.', price=340.0, url='https://www.dealnews.com/products/Small-Rig/Small-Rig-68-9-Carbon-Fiber-Video-Monopod/521836.html?iref=rss-c142'), Deal(product_description='The Jackery Explorer 1000 v2 portable power station packs a 1,069Wh LiFePO4 battery for extended off-grid power and can deliver up to 1,500W of AC output for camping or backup use. It includes a size M carrying case and multiple ports (USB-C, US

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


The SmallRig 68.9" Carbon Fiber Video Monopod is a rugged, lightweight support designed for stable handheld or mounted shots. It features a collapsible four-section design for compact transport, and dual 1/4-inch-20 and 3/8-inch-16 mounting compatibility to fit a wide range of cameras and accessories. It offers a one-button lift and lock height adjustment plus quick-release feet for rapid setup, enabling quick, confident adjustments on location. Built from carbon fiber, this monopod combines strength with lightness for demanding video work.
340.0
https://www.dealnews.com/products/Small-Rig/Small-Rig-68-9-Carbon-Fiber-Video-Monopod/521836.html?iref=rss-c142

The Jackery Explorer 1000 v2 portable power station packs a 1,069Wh LiFePO4 battery for extended off-grid power and can deliver up to 1,500W of AC output for camping or backup use. It includes a size M carrying case and multiple ports (USB-C, USB-A, DC, and AC) to recharge laptops, cameras, and other gear from a single source. The u

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='Carbon fiber monopod extends to 68.9 inches, providing lightweight stability for long shoots. The collapsible 4-section design folds for transport, while dual 1/4-20 and 3/8-16 mounting options support various cameras and accessories. It supports up to a 33-lb load and offers one-button height adjustment and quick-release feet for rapid setup.', price=340.0, url='https://www.dealnews.com/products/Small-Rig/Small-Rig-68-9-Carbon-Fiber-Video-Monopod/521836.html?iref=rss-c142'), Deal(product_description='Pixel Watch 3 comes in a 45mm matte black aluminum case with a large, high-contrast touchscreen. It delivers up to a full day of battery life with always-on display, and supports Wi‑Fi connectivity and fast charging for quick top-ups. The watch offers health tracking, notifications, and deep integration with Google services for a seamless Android experience. It is designed for everyday wear and reliable companion on the go.', price=126.0, url

### Optional: Pushover notifications

Pushover sends push notifications to your phone. Sign up at https://pushover.net/, create an application token, and add these values to the repository `.env` file:

```env
PUSHOVER_USER=u_...
PUSHOVER_TOKEN=a_...
```

Never commit the real values. After editing `.env`, restart the kernel and rerun the setup cell. Notification cells below safely skip sending when either value is missing.

In [14]:
# Reload the same repository .env after adding optional Pushover settings.
load_dotenv(dotenv_path, override=True)

True

In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
print("Pushover user configured:", bool(pushover_user))
print("Pushover token configured:", bool(pushover_token))

Pushover user configured: True
Pushover token configured: True


In [17]:
def push(message):
    if not pushover_user or not pushover_token:
        print(f"Pushover is not configured; skipped message: {message}")
        return False
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload, timeout=15)
    response.raise_for_status()
    print(f"Push sent: {message}")
    return True

In [25]:
# Sends only when both Pushover credentials are configured.
push("MASSIVE DEAL!!")

Push sent: MASSIVE DEAL!!


True

In [26]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent initialized with openai/gpt-5-nano
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


True

In [27]:
# This uses OpenAI to craft the message, then sends it only if Pushover is configured.
if pushover_user and pushover_token:
    agent.notify("A special deal on a Samsung 60-inch LED TV at a great bargain", 300, 1000, "https://www.samsung.com")
else:
    print("Skipping notify example because Pushover is not configured.")

INFO:root:[Messaging Agent] Messaging Agent is using openai/gpt-5-nano to craft the message
11:40:01 - LiteLLM:INFO: utils.py:4011 - 
LiteLLM completion() model= gpt-5-nano; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-5-nano; provider = openai
11:40:10 - LiteLLM:INFO: utils.py:1653 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
